

# Перевірка статиcтичних гіпотез. Z-test.



0. Зчитайте дані з `data.csv` у змінну data, яка має тип pandas.DataFrame. Ми будемо далі працювати з цією змінною.

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats import weightstats as stests

data = pd.read_csv('../data/data.csv')
data.head()

,Unnamed: 0,data
0,0,17.499453
1,1,19.662399
2,2,7.182823
3,3,29.841625
4,4,9.239386


Запустіть код нижче. Для коректної робити всіх подальших методів дані мають бути у вигляді одновимірного `numpy.ndarray` вектора та мати розмірність (100,). Така розмірність означає, що дані - одновимірні (колонка), якщо б розмірність була (100, 1), то дані сприймаються методами як двовимірні (таблиця), хоча для нас виглядати можуть так само.

In [2]:
data = data.data.values
data.shape

(100,)

## **Завдання 1**. (7 балів)
Зчитані дані - це сума покупок в доларах наших покупців на сайті протягом липня 2025 року.

До цього місяця, ми мали продажі в середньому на чек $20.

Необхідно зʼясувати, чи вийшло у нас статистично значущо **підвищити** середній чек за липень 2025?

Аби дати відповідь - ми проведемо z-test на рівні значущості $\alpha=0.05$ двома способами. В цьому завданні ми будемо виконувати обчислення "вручну" з використанням бібліотек numpy та scipy.stats подібно до прикладу в лекції. Для цього виконайте наступні 7 кроків. Правильне виконання кожного з кроків оцінюється в 1 бал.

1. Запишіть параметри задачі у змінні Python:
    - `sample_mean` - $\bar{x}$, середнє значення в вибірці
    - `population_mean` - $\mu_0$, середнє значення в популяції (тобто те, з яким ми порівнюємо середнє вибіркове значення)
    - `population_std` - $\sigma$, вибіркове стандартне відхилення, яке ми вважаємо, що є рівним ст. відх. популяції, адже маємо достатньо велику вибірку.
    - `sample_size` - $n$, розмір вибірки
    - `alpha` - $\alpha$ рівень значущості
    
    **Увага!** Для обчислення стандартного відхилення маємо скористатись функцією `np.std(your_dataframe, ddof=1)`. Чому так - розбираємо в лекції про t-test.


2. Визначте для цієї задачі:
    - якою є гіпотеза $H_0$
    - якою є альтернативна гіпотеза $H_a$
    - з яким типом тесту ми маємо справу - лівосторонній, правосторонній чи двосторонній.

3. Обчисліть стандартну помилку SE.
4. Розрахуйте z-статистику.
5. Знайдіть критичне z-значення з допомогою бібліотеки stats.
6. Обчисліть p-value з допомогою бібліотеки stats.
7. Прийміть рішення, чи відхиляєте ви гіпотезу $H_0$. Для прийняття рішення зробіть і порівняння z-статистики з критичним значенням, і проаналізуйте p-value.

### Задаємо параметри

In [3]:
sample_mean = data.mean()                         # x̄ - середнє значення в вибірці
population_mean = 20                              # μ₀ - середнє значення в популяції 
population_std = np.std(data, ddof=1)             # σ - вибіркове стандартне відхилення
sample_size = len(data)                           # n - розмір вибірки
alpha = 0.05                                      # α - рівень значущості (ймовірність помилки I роду)

### Гіпотези 

H₀: μ = 20 (середній чек продажів не змінився)  
H₁: μ > 20 (середній чек виріс)  
Тип тесту: правосторонній (one-tailed), оскільки альтернативна гіпотеза стверджує, що середній чек виріс (тобто знаходиться справа на графіку).  

### Розрахунок стандартної помилки SE

In [4]:
standard_error = population_std / np.sqrt(sample_size)

print("Формула: SE = σ / √n")
print(f"SE = {population_std:.3f} / √{sample_size}")
print(f"SE = {population_std:.3f} / {np.sqrt(sample_size):.3f}")
print(f"SE = {standard_error:.3f}")
print()
print(f"В середньому, вибіркове середнє відхиляється від справжнього середнього чека на ±{standard_error:.3f} $")

Формула: SE = σ / √n
SE = 6.254 / √100
SE = 6.254 / 10.000
SE = 0.625

В середньому, вибіркове середнє відхиляється від справжнього середнього чека на ±0.625 $


### Розрахунок z-статистики

In [5]:
z_statistic = (sample_mean - population_mean) / standard_error

print("Формула: z = (x̄ - μ₀) / SE")
print(f"z = ({sample_mean:.3f} - {population_mean}) / {standard_error:.3f}")
print(f"z = {z_statistic:.3f}")
print()
print(f"Наше вибіркове середнє відхиляється на {z_statistic:.3f} стандартних помилок від гіпотетичного середнього популяції.")
if z_statistic > 2:
    print("Це досить велике відхилення. (більше 2 стандартних помилок)")
elif z_statistic > 1:
    print("Це помірне відхилення (між 1 та 2 стандартними помилками)")
else:
    print("Це невелике відхилення (менше 1 стандартної помилки)")

Формула: z = (x̄ - μ₀) / SE
z = (19.378 - 20) / 0.625
z = -0.995

Наше вибіркове середнє відхиляється на -0.995 стандартних помилок від гіпотетичного середнього популяції.
Це невелике відхилення (менше 1 стандартної помилки)


### Критичне z-значення за допомогою бібліотеки stats

In [6]:
z_critical = stats.norm.ppf(1 - alpha)

print(f"Для рівня значущості α = {alpha} (правосторонній тест)")
print(f"Критичне z-значення = {z_critical:.3f}")
print()
print("Що це означає?")
print(f"Для правостороннього тесту при {alpha*100}% критичне значення = {z_critical:.3f}")
print(f"Якщо z-статистика > {z_critical:.3f}, то нульову гіпотезу відхиляємо.")
print(f"Ймовірність отримати z > {z_critical:.3f} випадково = {alpha*100}%")

Для рівня значущості α = 0.05 (правосторонній тест)
Критичне z-значення = 1.645

Що це означає?
Для правостороннього тесту при 5.0% критичне значення = 1.645
Якщо z-статистика > 1.645, то нульову гіпотезу відхиляємо.
Ймовірність отримати z > 1.645 випадково = 5.0%


### p-value за допомогою бібліотеки stats

In [7]:
p_value = 1 - stats.norm.cdf(z_statistic)

print(f"p-value = {p_value:.6f}")
print()
print("Що це означає?")
print(f"За умови, що H₀ правильна, ймовірність отримати z-статистику не меншу за спостережену становить {p_value:.6f}")
print(f"або приблизно {p_value*100:.4f}%")
if p_value < 0.001:
    print("Це НАДЗВИЧАЙНО малоймовірно! (менше 0.1%)")
elif p_value < 0.01:
    print("Це дуже малоймовірно! (менше 1%)")
elif p_value < 0.05:
    print("Це малоймовірно! (менше 5%)")
else:
    print("Це цілком можливо випадково")

p-value = 0.840216

Що це означає?
За умови, що H₀ правильна, ймовірність отримати z-статистику не меншу за спостережену становить 0.840216
або приблизно 84.0216%
Це цілком можливо випадково


### Рішення по гіпотезі
#### Порівняння z-статистики з критичним значенням

In [8]:
print("МЕТОД 1: Порівняння z-статистики з критичним значенням")
print(f"z-статистика = {z_statistic:.3f}")
print(f"Критичне значення = {z_critical:.3f}")
print(f"Порівняння: {z_statistic:.3f} {'>' if z_statistic > z_critical else '≤'} {z_critical:.3f}")

if z_statistic > z_critical:
    print("Висновок: z-статистика ПЕРЕВИЩУЄ критичне значення")
    decision1 = "ВІДХИЛЯЄМО H₀"
else:
    print("Висновок: z-статистика НЕ перевищує критичне значення")
    decision1 = "НЕ ВІДХИЛЯЄМО H₀"

print(f"Рішення: {decision1}")

МЕТОД 1: Порівняння z-статистики з критичним значенням
z-статистика = -0.995
Критичне значення = 1.645
Порівняння: -0.995 ≤ 1.645
Висновок: z-статистика НЕ перевищує критичне значення
Рішення: НЕ ВІДХИЛЯЄМО H₀


#### Аналіз p-value 

In [9]:
print("МЕТОД 2: Порівняння p-value з рівнем значущості")
print(f"p-value = {p_value:.6f}")
print(f"Рівень значущості α = {alpha}")
print(f"Порівняння: {p_value:.6f} {'<' if p_value < alpha else '≥'} {alpha}")

if p_value < alpha:
    print("Висновок: p-value МЕНШЕ за рівень значущості")
    decision2 = "ВІДХИЛЯЄМО H₀"
else:
    print("Висновок: p-value НЕ менше за рівень значущості")
    decision2 = "НЕ ВІДХИЛЯЄМО H₀"

print(f"Рішення: {decision2}")

МЕТОД 2: Порівняння p-value з рівнем значущості
p-value = 0.840216
Рівень значущості α = 0.05
Порівняння: 0.840216 ≥ 0.05
Висновок: p-value НЕ менше за рівень значущості
Рішення: НЕ ВІДХИЛЯЄМО H₀


In [10]:
print("\n" + "=" * 90)
print("ФІНАЛЬНИЙ ВИСНОВОК:")
print("=" * 90)
if decision2 == "ВІДХИЛЯЄМО H₀":
    print("✅ Твердження про ріст середнього чеку у липні 2025 року ПІДТВЕРДЖУЄТЬСЯ!")
    print("   Середній чек дійсно більший рівні значущості 5%")
    print(f"   Середній чек ({sample_mean}) статистично значуще")
    print(f"   перевищує середній чек у липні 2025 року ({population_mean})")
else:
    print("❌ Твердження про ріст середнього чеку у липні 2025 року НЕ ПІДТВЕРДЖУЄТЬСЯ")
    print(f' Оскільки вибіркове середнє {round(sample_mean,2)} менше середнього значення в популяції {population_mean} і')
    print(" немає достатніх статистичних доказів, що середній чек став більший у липні 2025 року")

print("\n" + "=" * 90)


ФІНАЛЬНИЙ ВИСНОВОК:
❌ Твердження про ріст середнього чеку у липні 2025 року НЕ ПІДТВЕРДЖУЄТЬСЯ
 Оскільки вибіркове середнє 19.38 менше середнього значення в популяції 20 і
 немає достатніх статистичних доказів, що середній чек став більший у липні 2025 року



## **Завдання 2.** (3 бали)
Виконайте обчислення z-test з використанням бібліотеки statsmodels.

Отримайте z-статистику та р-значення.

Виведіть p-значення та зробіть висновок, чи ми маємо достатньо доказів, аби стверджувати, що середній чек зріс.

Чи зійшлись значення z-статистики та р-значення в цьому завданні з попередніми обчисленнями?

In [11]:
# === СИНТАКСИС ФУНКЦІЇ === 
# stests.ztest(
                # x1,                         # масив даних для тестування 
                # x2=None,                    # другий масив (для двовибіркового тесту) 
                # value=0,                    # значення для порівняння (наше μ₀) 
                # alternative='two-sided',    # тип тесту 
                # usevar='pooled',            # як використовувати дисперсію - лише для двовибіркових тестів 
                # ddof=1.0)                   # ступені свободи, у даному завданні немає

print("ВИКОНАННЯ Z-ТЕСТУ з використанням бібліотеки statsmodels:") 
# Виконуємо правосторонній z-тест 
z_stat_sm, p_val_sm = stests.ztest(data,                     # наші дані 
                                   value = population_mean,  # μ₀ = 20 
                                   alternative = 'larger')   # H₁: μ > 20 

print(f"Результати z-тесту:") 
print(f" Z-статистика: {z_stat_sm:.3f}") 
print(f" P-value: {p_val_sm:.6f}") 
print() 

# Інтерпретація результатів print("ІНТЕРПРЕТАЦІЯ:") 
print("-" * 40) 
if p_val_sm < alpha: 
    print(f"✅ ВІДХИЛЯЄМО H₀") 
    print(f" p-value ({p_val_sm:.6f}) < α ({alpha})") 
    print(" Висновок: Середній чек статистично значуще вищий середнього чеку до липня") 
else: 
    print(f"❌ НЕ ВІДХИЛЯЄМО H₀") 
    print(f" p-value ({p_val_sm:.6f}) ≥ α ({alpha})") 
    print(" Висновок: Немає достатніх доказів про вищий середній чек у липні 2025") 

# Порівняння з ручними розрахунками 
print("\n" + "-" * 40) 
print("ПОРІВНЯННЯ З РУЧНИМИ РОЗРАХУНКАМИ:") 
print("-" * 40) 
print(f"Z-статистика:") 
print(f" Ручний розрахунок: {z_statistic:.3f}") 
print(f" Statsmodels: {z_stat_sm:.3f}") 
print(f" Різниця: {abs(z_statistic - z_stat_sm):.3f}") 
print("-" * 40) 
print("P-value:")
print(f"  Ручний розрахунок: {p_value:.6f}")
print(f"  Statsmodels:       {p_val_sm:.6f}")
print(f"  Різниця:           {abs(p_value - p_val_sm):.6f}")
print()
print("Примітка: Z-значення та p-value, розраховані обома методами, збіглися")
print("Отже, результати statsmodels повністю підтвердили ручні розрахунки.")

ВИКОНАННЯ Z-ТЕСТУ з використанням бібліотеки statsmodels:
Результати z-тесту:
 Z-статистика: -0.995
 P-value: 0.840216

----------------------------------------
❌ НЕ ВІДХИЛЯЄМО H₀
 p-value (0.840216) ≥ α (0.05)
 Висновок: Немає достатніх доказів про вищий середній чек у липні 2025

----------------------------------------
ПОРІВНЯННЯ З РУЧНИМИ РОЗРАХУНКАМИ:
----------------------------------------
Z-статистика:
 Ручний розрахунок: -0.995
 Statsmodels: -0.995
 Різниця: 0.000
----------------------------------------
P-value:
  Ручний розрахунок: 0.840216
  Statsmodels:       0.840216
  Різниця:           0.000000

Примітка: Z-значення та p-value, розраховані обома методами, збіглися
Отже, результати statsmodels повністю підтвердили ручні розрахунки.


**Time spent - 3 hour**

**What I learned**
- **Z-test:** перевіряти гіпотези щодо середнього значення.
- **Z-статистика:** показує, на скільки стандартних помилок вибіркове середнє відхиляється від гіпотетичного.
- **P-value:** ймовірність отримати такий результат за умови, що `H₀` правильна.
- **Python:** рахувати `SE`, `z-statistic`, `p-value` вручну та через `statsmodels`.
- **Interpretation:** робити висновок за `z`, `p-value` і рівнем значущості `α`.